In [12]:
# Import required libraries
from openai import OpenAI
import json
from PIL import Image
import base64
import os

In [9]:
api_key = os.environ["OPENAI_API_KEY"]

In [10]:
# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

In [13]:
client = OpenAI(api_key = api_key)

In [40]:
response_schema = {
  "type": "OBJECT",
  "properties": {
    "Activities": {
      "type": "ARRAY",
      "items": {
        "type": "OBJECT",
        "properties": {
          "Activity ID": {
            "type": "STRING"
          },
          "Question": {
            "type": "STRING",
          },
          "Score": {
            "type": "INT",
          },
          "Reasoning": {
            "type": "STRING",
          },
        },
        "required": ["Activity ID", "Question", "Score", "Reasoning"],
      },
    },
  },
  "required": ["Activities"],
}

In [46]:
# Function to process prompt
def evaluate_prompt(json_data, screenshot_path=None):
    if screenshot_path:
        base64_image = encode_image(screenshot_path)
    
    activity_id = json_data[0]['activityId']
    
    prompt_2 = f"""
        Analyse the activity’s page design based on the following, give a score (1-10) for each question:
        
    **Activity ID**:
    
    {activity_id}
        
    **Question**
        1. Is the text minimal and broken into digestible sections?
        2. Are images, dice, buttons, or other visuals prominently displayed above the fold?
        3. Are bullet points used over prose where appropriate?
        4. Does the page use sentence case formatting for titles and appropriate subheadings (e.g., 'You will need', 'How to play')?
        5. Is extra information provided in modals if necessary?
        Provide suggestions for improvement if any of these criteria are not met.
        
    
    **Score**
        1. Score (1-10)
        2. Score (1-10)
        3. Score (1-10)
        4. Score (1-10)
        5. Score (1-10)
        
        
    **Reasoning**
        1. Reason
        2. Reason
        3. Reason
        4. Reason
        5. Reason
    """
    
    
    combined_prompt = f"""
    The following JSON data represents educational activities on a webpage:
    {json.dumps(json_data, indent=2)}

    Use the json data and the encoded image to answer the following questions.

    {prompt_2}
    
    **Response Schema**
    
    {response_schema}
    
    **Remember:**
         
        * Respond in the relevant schema
        * Do not use markdown in your response
        * Answer all questions
        * Assume all videos are functional despite any privacy settings errors
    """
    
    response = client.chat.completions.create(
          model="gpt-4o-mini",
          messages=[
            {
              "role": "user",
              "content": [
                {
                  "type": "text",
                  "text": combined_prompt,
                },
                {
                  "type": "image_url",
                  "image_url": {
                    "url":  f"data:image/jpeg;base64,{base64_image}"
                  },
                },
              ],
            }
          ],
        )
    
    # response = openai.ChatCompletion.create(
    #     model="gpt-4",
    #     messages=[
    #         {"role": "system", "content": "You are a web content and educational guideline expert."},
    #         {"role": "user", "content": combined_prompt}
    #     ]
    # )
    
    return response.choices[0].message.content

In [32]:
# Load JSON data (replace with your file path)
with open('185986732.json', 'r') as file:
    json_data = json.load(file)

In [55]:
screenshot_path = "185986732.png"
response_2 = evaluate_prompt(json_data, screenshot_path)

In [56]:
data = json.loads(response_2)
data

{'Activities': [{'Activity ID': '185986732',
   'Question': 'Is the text minimal and broken into digestible sections?',
   'Score': 8,
   'Reasoning': 'The text is broken into clear challenges, making it easier to read, though some sections could be more concise.'},
  {'Activity ID': '185986732',
   'Question': 'Are images, dice, buttons, or other visuals prominently displayed above the fold?',
   'Score': 7,
   'Reasoning': 'The images are used but not prominently above the fold, as some content is pushed lower on the page.'},
  {'Activity ID': '185986732',
   'Question': 'Are bullet points used over prose where appropriate?',
   'Score': 6,
   'Reasoning': 'Some information could benefit from bullet points for clarity, particularly in listing instructions.'},
  {'Activity ID': '185986732',
   'Question': "Does the page use sentence case formatting for titles and appropriate subheadings (e.g., 'You will need', 'How to play')?",
   'Score': 9,
   'Reasoning': 'The titles and subheading

In [49]:
results = []

In [50]:
results.append(data['Activities'])

In [51]:
results

[[{'Activity ID': '185986732',
   'Question': 'Is the text minimal and broken into digestible sections?',
   'Score': 8,
   'Reasoning': 'The instructional text is clear and divided into challenges, making it easy to follow.'},
  {'Activity ID': '185986732',
   'Question': 'Are images, dice, buttons, or other visuals prominently displayed above the fold?',
   'Score': 7,
   'Reasoning': 'There are visuals related to the tasks, but some important information is not prominently visible at first glance.'},
  {'Activity ID': '185986732',
   'Question': 'Are bullet points used over prose where appropriate?',
   'Score': 9,
   'Reasoning': 'Bullet points are used to succinctly present the challenges and materials needed.'},
  {'Activity ID': '185986732',
   'Question': "Does the page use sentence case formatting for titles and appropriate subheadings (e.g., 'You will need', 'How to play')?",
   'Score': 6,
   'Reasoning': 'While there are headings, more emphasis on sentence case and consiste

In [53]:
import pandas as pd
results_df = pd.DataFrame(results[0])

In [54]:
results_df

,Activity ID,Question,Score,Reasoning
0,185986732,Is the text minimal and broken into digestible...,8,The instructional text is clear and divided in...
1,185986732,"Are images, dice, buttons, or other visuals pr...",7,"There are visuals related to the tasks, but so..."
2,185986732,Are bullet points used over prose where approp...,9,Bullet points are used to succinctly present t...
3,185986732,Does the page use sentence case formatting for...,6,"While there are headings, more emphasis on sen..."
4,185986732,Is extra information provided in modals if nec...,5,The modals contain some information but could ...


In [52]:
for item in results[0:]:
    new_rows_df = pd.DataFrame(item)
    results_df = pd.concat([results_df, new_rows_df], ignore_index=True)

NameError: name 'results_df' is not defined